# Personal Knowledge Worker
A RAG chatbot that answers questions from your own markdown files.

**Setup:** Add `.md` files to the `knowledge-base/` folder, then run all cells.

In [ ]:
import os
import glob
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from chromadb import PersistentClient

load_dotenv(override=True)
client = OpenAI()

LLM = 'gpt-4o-mini'
EMBEDDING_MODEL = 'text-embedding-3-small'
KB_PATH = 'knowledge-base'
DB_NAME = 'knowledge_db'
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80

In [ ]:
# --- Load and chunk markdown files ---

def load_documents():
    docs = []
    for path in glob.glob(f'{KB_PATH}/**/*.md', recursive=True):
        with open(path, 'r', encoding='utf-8') as f:
            docs.append({'source': path, 'text': f.read()})
    print(f'Loaded {len(docs)} documents')
    return docs

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

def build_chunks(docs):
    all_chunks = []
    for doc in docs:
        for chunk in chunk_text(doc['text']):
            all_chunks.append({'text': chunk, 'source': doc['source']})
    print(f'Created {len(all_chunks)} chunks')
    return all_chunks

docs = load_documents()
chunks = build_chunks(docs)

In [ ]:
# --- Embed and store in ChromaDB ---

def build_vectorstore(chunks):
    texts = [c['text'] for c in chunks]
    metas = [{'source': c['source']} for c in chunks]

    embeddings = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    vectors = [e.embedding for e in embeddings.data]

    db = PersistentClient(path=DB_NAME)
    if 'knowledge' in [c.name for c in db.list_collections()]:
        db.delete_collection('knowledge')
    collection = db.get_or_create_collection('knowledge')

    ids = [str(i) for i in range(len(chunks))]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f'Vector store ready with {collection.count()} chunks')
    return collection

collection = build_vectorstore(chunks)

In [ ]:
# --- RAG chat ---

SYSTEM = """You are a helpful personal assistant with access to the user's knowledge base.
Answer questions accurately using only the provided context.
If the answer isn't in the context, say you don't know."""

def retrieve(question, k=5):
    query_vec = client.embeddings.create(model=EMBEDDING_MODEL, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query_vec], n_results=k)
    return '\n\n'.join(results['documents'][0])

def chat(message, history):
    context = retrieve(message)
    messages = [
        {"role": "system", "content": SYSTEM + f"\n\nContext:\n{context}"}
    ] + history + [
        {"role": "user", "content": message}
    ]
    response = client.chat.completions.create(model=LLM, messages=messages, stream=True)
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

gr.ChatInterface(
    fn=chat,
    type='messages',
    title='Personal Knowledge Worker',
    description='Ask me anything about your knowledge base.'
).launch()